In [ ]:
import pandas as pd
import re
from google.colab import drive

!pip install transformers torch -q

from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier",
    device=0
)

# mount drive
drive.mount('/content/drive')

# load data
df_kompas = pd.read_csv("/content/drive/MyDrive/BIG DATA/kompas.csv", sep=';')
df_detik = pd.read_csv("/content/drive/MyDrive/BIG DATA/detik.csv", sep=';')
df_liputan6 = pd.read_csv("/content/drive/MyDrive/BIG DATA/liputan6.csv", sep=';')
df_kaggle = pd.read_csv("/content/drive/MyDrive/BIG DATA/kaggle.csv")

# kolom sumber
df_kompas["Sumber"] = "Kompas"
df_detik["Sumber"] = "Detik"
df_liputan6["Sumber"] = "Liputan6"

df_kaggle.rename(columns={
    "Content": "Isi",
    "Waktu": "Tanggal"
}, inplace=True)

df_kaggle["Link"] = "-"
df_kaggle["Sumber"] = "Kaggle"

# gabung scraping
df_scraping = pd.concat([df_kompas, df_detik, df_liputan6], ignore_index=True)

# gabung all data
df_all = pd.concat([df_scraping, df_kaggle], ignore_index=True)

# cleaning
df_all = df_all.drop_duplicates(subset=["Judul"])
df_all = df_all.dropna(subset=["Judul", "Isi"])

df_all["Isi"] = df_all["Isi"].str.replace("\n", " ")
df_all["Isi"] = df_all["Isi"].str.strip()
df_all["Isi"] = df_all["Isi"].str.lower()
df_all["Isi"] = df_all["Isi"].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x))
df_all["Isi"] = df_all["Isi"].apply(lambda x: re.sub(r'\s+', ' ', x))

df_all = df_all.reset_index(drop=True)

#Transformasi format tanggal
df_all["Tanggal"] = pd.to_datetime(df_all["Tanggal"], errors='coerce')

# pengambilan kolom
df_all = df_all[["Judul", "Isi", "Tanggal", "Link", "Sumber"]]

# sentimen
print("Sedang proses analisis sentimen...")

texts= df_all["Isi"].astype(str).tolist()

hasil = classifier(
    texts,
    batch_size=32,
    truncation=True,
    max_length=512
)

# mapping hasil
sentimen = []
for h in hasil:
  if h['label'] == "positive":
    sentimen.append("Positif")
  elif h['label'] == "negative":
    sentimen.append("Negatif")
  else:
    sentimen.append("Netral")

df_all["Sentimen"] = sentimen

print("\nCek jumlah data:")
print(len(df_all), "=", len(sentimen))

print("\nDistribusi Sentimen:")
print(df_all["Sentimen"].value_counts())

# simpan
df_all.to_csv("/content/drive/MyDrive/BIG DATA/dataset_final_with_sentimen.csv", index=False)

print("SELESAI")




Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: w11wo/indonesian-roberta-base-sentiment-classifier
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_6683/3500658679.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_all["Tanggal"] = pd.to_datetime(df_all["Tanggal"], errors='coerce')


Sedang proses analisis sentimen...
